### 기본 라이브러리

In [46]:
import pandas as pd
import numpy as np
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import time

In [ ]:
# ./corr/모든_상관계수_결측치_제거.csv
# 이 파일 안에 있는 Variable과 일치하는 컬럼명을 찾아서, 원본 데이터의 값을 가져온다.

##### Variable 컬럼명만 남기기

In [7]:
correlation = pd.read_csv('./corr/모든_상관계수_결측치_제거.csv', encoding='utf-8-sig')

In [8]:
correlation

,Variable,Correlation
0,이용금액_R3M_신용체크,-6.228267e-01
1,청구금액_R6M,-5.979035e-01
2,청구금액_R3M,-5.906781e-01
3,이용금액_R3M_신용,-5.890316e-01
4,청구금액_B0,-5.779884e-01
...,...,...
710,컨택건수_이용유도_인터넷_B0M,2.260987e-03
711,컨택건수_CA_청구서_B0M,1.259665e-03
712,컨택건수_보험_TM_B0M,-9.574610e-04
713,컨택건수_CA_TM_R6M,-3.480850e-04


In [9]:
corr_df = correlation[correlation['Correlation'].abs() >= 0.4]
corr_df

,Variable,Correlation
0,이용금액_R3M_신용체크,-0.622827
1,청구금액_R6M,-0.597904
2,청구금액_R3M,-0.590678
3,이용금액_R3M_신용,-0.589032
4,청구금액_B0,-0.577988
5,_1순위카드이용금액,-0.573870
6,평잔_일시불_6M,-0.447967
7,월중평잔_일시불_B0M,-0.444621
8,월중평잔_일시불,-0.441527
9,평잔_일시불_3M,-0.434514


In [22]:
# variable_df = corr_df['Variable']
# variable_df

##### 다중공선성 계산하기

./data/train/1_회원정보_train.parquet
./data/train/2_신용정보_train.parquet
./data/train/3_승인매출정보_train.parquet
./data/train/4_청구입금정보_train.parquet
./data/train/5_잔액정보_train.parquet
./data/train/6_채널정보_train.parquet
./data/train/7_마케팅정보_train.parquet
./data/train/8_성과정보_train.parquet

In [24]:
selected_columns = corr_df['Variable'].tolist()

# 데이터 경로 목록
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

# 필요한 컬럼들을 담을 리스트
df_list = []

for path in paths:
    df = pd.read_parquet(path)
    # 현재 파일에서 필요한 컬럼만 추출
    common_cols = list(set(df.columns) & set(selected_columns))
    if common_cols:
        df_list.append(df[common_cols])

# 컬럼 기준으로 병합 (axis=1)
merged_df = pd.concat(df_list, axis=1)

In [32]:
corr_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 250
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Variable     50 non-null     object 
 1   Correlation  50 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.2+ KB


In [28]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400000 entries, 0 to 2399999
Data columns (total 50 columns):
 #   Column           Dtype
---  ------           -----
 0   이용금액_R3M_신용      int64
 1   이용금액_R3M_신용체크    int64
 2   _2순위카드이용금액       int64
 3   _1순위카드이용금액       int64
 4   최대이용금액_일시불_R12M  int64
 5   이용건수_일시불_R6M     int64
 6   정상청구원금_B2M       int64
 7   연체입금원금_B0M       int64
 8   이용건수_일시불_R12M    int64
 9   _2순위업종_이용금액      int64
 10  이용건수_오프라인_R3M    int64
 11  이용금액_일시불_R12M    int64
 12  이용금액_일시불_R6M     int64
 13  이용건수_오프라인_B0M    int64
 14  이용가맹점수           int64
 15  이용금액_일시불_R3M     int64
 16  _3순위업종_이용금액      int64
 17  정상청구원금_B0M       int64
 18  _2순위쇼핑업종_이용금액    int64
 19  _1순위교통업종_이용금액    int64
 20  이용건수_신판_R3M      int64
 21  이용건수_신용_R12M     int64
 22  이용금액_오프라인_B0M    int64
 23  이용건수_오프라인_R6M    int64
 24  이용건수_신용_R3M      int64
 25  이용금액_일시불_B0M     int64
 26  이용건수_일시불_R3M     int64
 27  _1순위업종_이용금액      int64
 28  이용건수_신판_B0M      int64
 29  정상입금원금_B2M    

In [36]:
import statsmodels.api as sm

def calculate_vif(df):
    df_with_const = sm.add_constant(df)
    vif_data = pd.DataFrame()
    vif_data['Variable'] = df_with_const.columns
    vif_data['VIF'] = [variance_inflation_factor(df_with_const.values, i) for i in range(df_with_const.shape[1])]
    return vif_data  # const 포함

In [37]:
import time

current_df = merged_df
columns_by_step = {}
vif_logs = []

while current_df.shape[1] > 5:
    start_time = time.time()  # 시작 시간
    
    vif_result = calculate_vif(current_df)
    vif_result_sorted = vif_result.sort_values('VIF', ascending=False)
    
    max_vif_variable = vif_result_sorted.iloc[0]['Variable']
    max_vif_value = vif_result_sorted.iloc[0]['VIF']
    
    columns_by_step[current_df.shape[1]] = current_df.columns.tolist()
    vif_logs.append({
        '컬럼 개수': current_df.shape[1],
        '제거된 컬럼': max_vif_variable,
        '해당 VIF': max_vif_value
    })
    
    elapsed = time.time() - start_time  # 경과 시간
    print(f"[{current_df.shape[1]} → {current_df.shape[1]-1}] 제거: {max_vif_variable}, VIF: {max_vif_value:.2f}, 소요 시간: {elapsed:.2f}초")
    
    current_df = current_df.drop(columns=[max_vif_variable])

# 마지막 5개도 저장
columns_by_step[5] = current_df.columns.tolist()

[50 → 49] 제거: 이용건수_신판_R6M, VIF: 107585.34, 소요 시간: 303.41초
[49 → 48] 제거: 이용건수_신판_R3M, VIF: 63224.25, 소요 시간: 286.63초
[48 → 47] 제거: 이용건수_신판_R12M, VIF: 34029.98, 소요 시간: 330.79초
[47 → 46] 제거: 이용건수_일시불_R3M, VIF: 20672.46, 소요 시간: 329.06초
[46 → 45] 제거: 이용건수_신용_R6M, VIF: 12333.79, 소요 시간: 291.65초
[45 → 44] 제거: 이용건수_신판_B0M, VIF: 10271.53, 소요 시간: 244.34초
[44 → 43] 제거: 이용건수_일시불_R12M, VIF: 4881.35, 소요 시간: 239.44초
[43 → 42] 제거: 이용건수_신용_B0M, VIF: 2564.62, 소요 시간: 256.22초
[42 → 41] 제거: 이용건수_신용_R3M, VIF: 134.23, 소요 시간: 255.50초
[41 → 40] 제거: 이용금액_일시불_R3M, VIF: 86.42, 소요 시간: 229.42초
[40 → 39] 제거: 이용금액_R3M_신용, VIF: 83.22, 소요 시간: 196.02초
[39 → 38] 제거: 이용건수_일시불_R6M, VIF: 73.60, 소요 시간: 180.36초
[38 → 37] 제거: 이용금액_일시불_B0M, VIF: 56.34, 소요 시간: 168.18초
[37 → 36] 제거: 정상청구원금_B0M, VIF: 52.18, 소요 시간: 158.16초
[36 → 35] 제거: 이용건수_일시불_B0M, VIF: 33.01, 소요 시간: 146.06초
[35 → 34] 제거: 이용건수_오프라인_R3M, VIF: 32.80, 소요 시간: 138.86초
[34 → 33] 제거: 월중평잔_일시불, VIF: 30.28, 소요 시간: 134.67초
[33 → 32] 제거: 청구금액_R3M, VIF: 25.65, 소요 시간: 125.44초
[

In [40]:
# 딕셔너리를 DataFrame으로 변환 (빈칸은 NaN)
columns_step_df = pd.DataFrame.from_dict(columns_by_step, orient='index').T
columns_step_df = columns_step_df.sort_index(axis=1, ascending=False)  # [50→5 순서로 정렬]

columns_step_df.to_csv("컬럼_감소_과정.csv", index=False, encoding='utf-8-sig')

In [41]:
vif_log_df = pd.DataFrame(vif_logs)
vif_log_df.to_csv("제거_로그.csv", index=False, encoding='utf-8-sig')